# **Gráficos sobre Peligro Esperado (xT) en el Fútbol**

Este notebook forma parte de **Patrones Lab**, una serie de proyectos pensados para explorar datos reales con un enfoque claro, visual y reproducible. En este caso, el trabajo se centra en la base pública de **StatsBombpy (libería de Python)**, con el objetivo de ordenar los datos, revisar su calidad, construir variables útiles y empezar a detectar patrones dentro del futbol.

Cargo librerías y resultados procesados.


In [81]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go

ruta_processed = Path("../data/processed")

df_valor_zonas = pd.read_parquet(ruta_processed / "pe5_valor_zonas.parquet")

df_acciones_valoradas = pd.read_parquet(ruta_processed / "pe5_acciones_valoradas.parquet")

df_partidos_mundiales = pd.read_parquet(ruta_processed / "partidos_mundiales.parquet")

In [82]:
import warnings
warnings.filterwarnings("ignore")

In [83]:
df_partidos_mundiales[
    (df_partidos_mundiales["mundial"] == "Qatar 2022") &
    (df_partidos_mundiales["competition_stage"] == "Final")][
        ["match_id", "match_date", "home_team", "away_team", "home_score", "away_score", "competition_stage"]]

,match_id,match_date,home_team,away_team,home_score,away_score,competition_stage
73,3869685,2022-12-18,Argentina,France,3,3,Final


Identifico el partido de la final.


In [84]:
match_id_final = df_partidos_mundiales.loc[
    (df_partidos_mundiales["mundial"] == "Qatar 2022") &
    (df_partidos_mundiales["competition_stage"] == "Final"),
    "match_id"].iloc[0]

Filtro las acciones valoradas de la final.


In [85]:
df_final = df_acciones_valoradas[df_acciones_valoradas["match_id"] == match_id_final].copy()


Calculo el ranking PE5 de Argentina.


In [86]:
df_arg_final = df_final[df_final["team"] == "Argentina"].copy()

ranking_arg = (
    df_arg_final
    .groupby("player")
    .agg(
        acciones=("incremento_pe5", "count"),
        pe5_total=("incremento_pe5", "sum"),
        pe5_positivo=("pe5_positivo", "sum"),
        pe5_negativo=("pe5_negativo", "sum")
    )
    .reset_index()
    .sort_values("pe5_total", ascending=False)
)

ranking_arg_top = ranking_arg.head(10).sort_values("pe5_total")


Preparo el ranking completo de Argentina.


In [87]:
ranking_plot = (
    ranking_arg
    .copy()
    .sort_values("pe5_total", ascending=True)
    .reset_index(drop=True)
)

top_3 = ranking_plot.nlargest(3, "pe5_total")["player"].tolist()

ranking_plot[["player", "acciones", "pe5_total", "pe5_positivo", "pe5_negativo"]]


,player,acciones,pe5_total,pe5_positivo,pe5_negativo
0,Nicolás Hernán Otamendi,122,-0.005383,0.027672,-0.033054
1,Damián Emiliano Martínez,29,-0.000727,0.003524,-0.004251
2,Paulo Bruno Exequiel Dybala,4,-0.000396,0.000000,-0.000396
3,Germán Alejandro Pezzella,2,0.000832,0.000832,0.000000
4,Nicolás Alejandro Tagliafico,75,0.012149,0.066584,-0.054435
5,Cristian Gabriel Romero,99,0.014252,0.027855,-0.013603
6,Leandro Daniel Paredes,24,0.030237,0.033147,-0.002910
7,Lautaro Javier Martínez,12,0.033775,0.066956,-0.033181
8,Nahuel Molina Lucero,62,0.049498,0.076963,-0.027465
9,Enzo Fernandez,150,0.052560,0.100802,-0.048241


Configuro escala y colores del ranking argentino.


In [88]:
alto_base = 260
alto_por_jugador = 46
alto_grafico = alto_base + alto_por_jugador * len(ranking_plot)

x_min = min(0, ranking_plot["pe5_total"].min() * 1.15)
x_max = ranking_plot["pe5_total"].max() * 1.18

colores_barras = []
colores_label_bg = []
colores_label_border = []
colores_label_text = []

for player, valor in zip(ranking_plot["player"], ranking_plot["pe5_total"]):

    if player == top_3[0]:
        colores_barras.append("#FFD166")
        colores_label_bg.append("rgba(255, 209, 102, 0.22)")
        colores_label_border.append("#FFD166")
        colores_label_text.append("#FFF7E6")

    elif player in top_3:
        colores_barras.append("#5EC8F8")
        colores_label_bg.append("rgba(94, 200, 248, 0.20)")
        colores_label_border.append("#5EC8F8")
        colores_label_text.append("#EAF8FF")

    elif valor < 0:
        colores_barras.append("#EF476F")
        colores_label_bg.append("rgba(239, 71, 111, 0.13)")
        colores_label_border.append("rgba(239, 71, 111, 0.55)")
        colores_label_text.append("#FDE2EA")

    else:
        colores_barras.append("#3E6B89")
        colores_label_bg.append("rgba(255, 255, 255, 0.07)")
        colores_label_border.append("rgba(255, 255, 255, 0.22)")
        colores_label_text.append("#F8FAFC")


Grafico PE5 total por jugador argentino.


In [89]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=ranking_plot["pe5_total"],
        y=ranking_plot["player"],
        orientation="h",
        marker=dict(
            color=colores_barras,
            line=dict(
                color="rgba(255,255,255,0.35)",
                width=1.3 )),
        text=ranking_plot["pe5_total"],
        textposition="outside",
        texttemplate="<b>%{text:.3f}</b>",
        textfont=dict(
            size=18,
            color="#F8FAFC",
            family="Outfit, Arial, sans-serif"),
        cliponaxis=False,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "PE5 total: %{x:.3f}<br>"
            "<extra></extra>")))

fig.add_vline(
    x=0,
    line_width=1.2,
    line_color="rgba(255,255,255,0.30)")

fig.update_layout(
    title=dict(
        text="<b>Argentina — PE5 por jugador en la Final del Mundial</b>",
        x=0.5,
        xanchor="center",
        font=dict(
            size=38,
            color="#F8FAFC",
            family="Outfit, Arial, sans-serif" )),
    paper_bgcolor="#061327",
    plot_bgcolor="#0B1B33",
    font=dict(
        color="#E5E7EB",
        family="Outfit, Arial, sans-serif"),
    height=alto_grafico,
    margin=dict(l=380,r=120,t=110,b=85),
    bargap=0.28,
    showlegend=False)

fig.update_xaxes(
    title_text="<b>PE5 total acumulado</b>",
    range=[x_min, x_max],
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    ticks="outside",
    tickfont=dict(
        size=16,
        color="#CBD5E1",
        family="Outfit, Arial, sans-serif"),
    title_font=dict(
        size=15,
        color="#F8FAFC",
        family="Outfit, Arial, sans-serif"))

fig.update_yaxes(
    title_text="",
    showgrid=False,
    showticklabels=False)

for i, row in ranking_plot.iterrows():

    jugador = row["player"]
    valor = row["pe5_total"]

    if jugador == top_3[0]:
        texto = f"<b>★ {jugador}</b>"
    elif jugador in top_3:
        texto = f"<b>● {jugador}</b>"
    else:
        texto = f"<b>{jugador}</b>"

    fig.add_annotation(
        x=-0.15,
        y=jugador,
        xref="paper",
        yref="y",
        text=texto,
        showarrow=False,
        xanchor="right",
        yanchor="middle",
        align="right",
        font=dict(
            size=18,
            color=colores_label_text[i],
            family="Outfit, Arial, sans-serif"),
        bgcolor=colores_label_bg[i],
        bordercolor=colores_label_border[i],
        borderwidth=1.2,
        borderpad=6)

fig.add_annotation(
    x=0.5,
    y=-0.10,
    xref="paper",
    yref="paper",
    text="PE5 total = valor de zona destino - valor de zona origen, acumulado por jugador",
    showarrow=False,
    font=dict(
        size=14,
        color="#9FB3C8",
        family="Outfit, Arial, sans-serif"),
    align="center")

fig.show()


Ordeno Argentina para balance positivo y negativo.


In [90]:
ranking_arg_balance = (
    ranking_arg
    .copy()
    .sort_values("pe5_total", ascending=True)
    .reset_index(drop=True))


Grafico PE5 positivo y negativo de Argentina.


In [91]:
ranking_plot = ranking_arg_balance.copy()

top_3 = ranking_plot.nlargest(3, "pe5_total")["player"].tolist()

alto_base = 280
alto_por_jugador = 48
alto_grafico = alto_base + alto_por_jugador * len(ranking_plot)

colores_label_bg = []
colores_label_border = []
colores_label_text = []

for player in ranking_plot["player"]:
    if player == top_3[0]:
        colores_label_bg.append("rgba(255, 209, 102, 0.22)")
        colores_label_border.append("#FFD166")
        colores_label_text.append("#FFF7E6")
    elif player in top_3:
        colores_label_bg.append("rgba(94, 200, 248, 0.22)")
        colores_label_border.append("#5EC8F8")
        colores_label_text.append("#EAF8FF")
    else:
        colores_label_bg.append("rgba(255, 255, 255, 0.07)")
        colores_label_border.append("rgba(255, 255, 255, 0.25)")
        colores_label_text.append("#F8FAFC")

x_min = ranking_plot["pe5_negativo"].min() * 1.55
x_max = ranking_plot["pe5_positivo"].max() * 1.30

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=ranking_plot["player"],
        x=ranking_plot["pe5_negativo"],
        orientation="h",
        name="PE5 negativo",
        marker=dict(
            color="#EF476F",
            line=dict(color="rgba(255,255,255,0.35)", width=1.4)
        ),
        text=ranking_plot["pe5_negativo"],
        texttemplate="<b>%{text:.3f}</b>",
        textposition="outside",
        textfont=dict(
            size=24,
            color="#FFFFFF",
            family="Outfit, Arial, sans-serif"
        ),
        cliponaxis=False,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "PE5 negativo: %{x:.3f}<extra></extra>")))

fig.add_trace(
    go.Bar(
        y=ranking_plot["player"],
        x=ranking_plot["pe5_positivo"],
        orientation="h",
        name="PE5 positivo",
        marker=dict(
            color="#5EC8F8",
            line=dict(color="rgba(255,255,255,0.35)", width=1.4)
        ),
        text=ranking_plot["pe5_positivo"],
        texttemplate="<b>%{text:.3f}</b>",
        textposition="outside",
        textfont=dict(
            size=12,
            color="#FFFFFF",
            family="Outfit, Arial, sans-serif"
        ),
        cliponaxis=False,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "PE5 positivo: %{x:.3f}<extra></extra>")))

fig.add_vline(
    x=0,
    line_width=1.6,
    line_color="rgba(255,255,255,0.50)")

for i, row in ranking_plot.iterrows():
    jugador = row["player"]

    if jugador == top_3[0]:
        texto = f"<b>★ {jugador}</b>"
    elif jugador in top_3:
        texto = f"<b>● {jugador}</b>"
    else:
        texto = f"<b>{jugador}</b>"

    fig.add_annotation(
        x=-0.10,
        y=jugador,
        xref="paper",
        yref="y",
        text=texto,
        showarrow=False,
        xanchor="right",
        yanchor="middle",
        align="right",
        font=dict(
            size=18,
            color=colores_label_text[i],
            family="Outfit, Arial, sans-serif"
        ),
        bgcolor=colores_label_bg[i],
        bordercolor=colores_label_border[i],
        borderwidth=1.3,
        borderpad=6)

fig.update_layout(
    title=dict(
        text="<b>Argentina — xT generado y xT reducido por jugador</b>",
        x=0.5,
        xanchor="center",
        font=dict(
            family="Outfit, Arial, sans-serif",
            size=38,
            color="#F8FAFC")),
    paper_bgcolor="#061327",
    plot_bgcolor="#0B1B33",
    font=dict(
        color="#E5E7EB",
        family="Outfit, Arial, sans-serif"),
    barmode="relative",
    height=alto_grafico,
    margin=dict(l=380, r=110, t=110, b=90),
    bargap=0.26,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(0,0,0,0)",
        font=dict(
            size=13,
            color="#E5E7EB",
            family="Outfit, Arial, sans-serif")),
    uniformtext=dict(
        minsize=11,
        mode="show"))

fig.update_xaxes(
    title_text="<b>PE5 acumulado</b>",
    range=[x_min, x_max],
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    ticks="outside",
    tickfont=dict(
        size=18,
        color="#CBD5E1",
        family="Outfit, Arial, sans-serif"
    ),
    title_font=dict(
        size=15,
        color="#F8FAFC",
        family="Outfit, Arial, sans-serif"))

fig.update_yaxes(
    title_text="",
    showgrid=False,
    showticklabels=False)

fig.add_annotation(
    x=0.5,
    y=-0.12,
    xref="paper",
    yref="paper",
    text=(
        "PE5 positivo: acciones que aumentan peligro zonal · "
        "PE5 negativo: acciones que reducen peligro zonal"),
    showarrow=False,
    font=dict(
        size=14,
        color="#9FB3C8",
        family="Outfit, Arial, sans-serif"),
    align="center")

fig.show()


Calculo el ranking PE5 de Francia.


In [92]:
df_fra_final = df_final[
    df_final["team"] == "France"
].copy()

ranking_fra = (
    df_fra_final
    .groupby("player")
    .agg(
        acciones=("incremento_pe5", "count"),
        pe5_total=("incremento_pe5", "sum"),
        pe5_positivo=("pe5_positivo", "sum"),
        pe5_negativo=("pe5_negativo", "sum")
    )
    .reset_index()
    .sort_values("pe5_total", ascending=False))

ranking_fra


,player,acciones,pe5_total,pe5_positivo,pe5_negativo
9,Kylian Mbappé Lottin,54,0.304126,0.324146,-0.020021
1,Antoine Griezmann,35,0.204735,0.224787,-0.020052
13,Randal Kolo Muani,24,0.112679,0.140654,-0.027976
6,Ibrahima Konaté,3,0.093498,0.093498,0.000000
10,Marcus Thuram,35,0.073352,0.091714,-0.018361
7,Jules Koundé,89,0.056096,0.081395,-0.025299
15,Theo Bernard François Hernández,71,0.052767,0.060508,-0.007741
14,Raphaël Varane,100,0.038527,0.057985,-0.019458
4,Eduardo Camavinga,37,0.033902,0.047410,-0.013508
5,Hugo Lloris,33,0.027934,0.032288,-0.004354


Grafico PE5 total por jugador francés.


In [93]:
ranking_plot = (
    ranking_fra
    .copy()
    .sort_values("pe5_total", ascending=True)
    .reset_index(drop=True))

top_3 = ranking_plot.nlargest(3, "pe5_total")["player"].tolist()

alto_base = 260
alto_por_jugador = 48
alto_grafico = alto_base + alto_por_jugador * len(ranking_plot)

x_min = min(0, ranking_plot["pe5_total"].min() * 1.20)
x_max = ranking_plot["pe5_total"].max() * 1.22

azul_francia = "#0055A4"
blanco_francia = "#F8FAFC"
rojo_francia = "#EF4135"

colores_barras = []
colores_label_bg = []
colores_label_border = []
colores_label_text = []

for player, valor in zip(ranking_plot["player"], ranking_plot["pe5_total"]):

    if player == top_3[0]:
        colores_barras.append(rojo_francia)
        colores_label_bg.append("rgba(239, 65, 53, 0.22)")
        colores_label_border.append(rojo_francia)
        colores_label_text.append("#FFE8E6")

    elif player == top_3[1]:
        colores_barras.append(blanco_francia)
        colores_label_bg.append("rgba(248, 250, 252, 0.14)")
        colores_label_border.append(blanco_francia)
        colores_label_text.append("#F8FAFC")

    elif player == top_3[2]:
        colores_barras.append(azul_francia)
        colores_label_bg.append("rgba(0, 85, 164, 0.30)")
        colores_label_border.append("#5EC8F8")
        colores_label_text.append("#EAF6FF")

    elif valor < 0:
        colores_barras.append("#7F1D1D")
        colores_label_bg.append("rgba(239, 65, 53, 0.12)")
        colores_label_border.append("rgba(239, 65, 53, 0.45)")
        colores_label_text.append("#FDE2E0")

    else:
        colores_barras.append("#1E3A8A")
        colores_label_bg.append("rgba(0, 85, 164, 0.18)")
        colores_label_border.append("rgba(94, 200, 248, 0.35)")
        colores_label_text.append("#F8FAFC")

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=ranking_plot["pe5_total"],
        y=ranking_plot["player"],
        orientation="h",
        marker=dict(
            color=colores_barras,
            line=dict(
                color="rgba(255,255,255,0.45)",
                width=1.4
            )
        ),
        text=ranking_plot["pe5_total"],
        textposition="outside",
        texttemplate="<b>%{text:.3f}</b>",
        textfont=dict(
            size=16,
            color="#F8FAFC",
            family="Outfit, Arial, sans-serif"
        ),
        cliponaxis=False,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "PE5 total: %{x:.3f}<br>"
            "<extra></extra>")))

fig.add_vline(
    x=0,
    line_width=1.3,
    line_color="rgba(255,255,255,0.35)")

fig.update_layout(
    title=dict(
        text="<b>Francia — PE5 por jugador en la Final del Mundial</b>",
        x=0.5,
        xanchor="center",
        font=dict(
            family="Outfit, Arial, sans-serif",
            size=36,
            color="#F8FAFC")),
    paper_bgcolor="#061327",
    plot_bgcolor="#0B1B33",
    font=dict(
        color="#E5E7EB",
        family="Outfit, Arial, sans-serif"),
    height=alto_grafico,
    margin=dict(l=380,r=120,t=110,b=90),
    bargap=0.28,
    showlegend=False)

fig.update_xaxes(
    title_text="<b>PE5 total acumulado</b>",
    range=[x_min, x_max],
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    ticks="outside",
    tickfont=dict(
        size=16,
        color="#CBD5E1",
        family="Outfit, Arial, sans-serif"),
    title_font=dict(
        size=18,
        color="#F8FAFC",
        family="Outfit, Arial, sans-serif"))

fig.update_yaxes(
    title_text="",
    showgrid=False,
    showticklabels=False)

for i, row in ranking_plot.iterrows():

    jugador = row["player"]

    if jugador == top_3[0]:
        texto = f"<b>★ {jugador}</b>"
    elif jugador in top_3:
        texto = f"<b>● {jugador}</b>"
    else:
        texto = f"<b>{jugador}</b>"

    fig.add_annotation(
        x=-0.10,
        y=jugador,
        xref="paper",
        yref="y",
        text=texto,
        showarrow=False,
        xanchor="right",
        yanchor="middle",
        align="right",
        font=dict(
            size=18,
            color=colores_label_text[i],
            family="Outfit, Arial, sans-serif"),
        bgcolor=colores_label_bg[i],
        bordercolor=colores_label_border[i],
        borderwidth=1.3,
        borderpad=6)

fig.add_annotation(
    x=0.5,
    y=-0.09,
    xref="paper",
    yref="paper",
    text="PE5 total = valor de zona destino - valor de zona origen, acumulado por jugador",
    showarrow=False,
    font=dict(
        size=16,
        color="#9FB3C8",
        family="Outfit, Arial, sans-serif"),
    align="center")

fig.show()


Descargo eventos completos de la final.


In [94]:
from statsbombpy import sb

df_eventos_final = sb.events(match_id=match_id_final)

df_eventos_final.shape


(4407, 94)

In [95]:
df_eventos_final[
    ["match_id", "period", "minute", "second", "team", "type", "player", "shot_type", "shot_outcome"]
].head()


,match_id,period,minute,second,team,type,player,shot_type,shot_outcome
0,3869685,1,0,0,Argentina,Starting XI,NaN,NaN,NaN
1,3869685,1,0,0,France,Starting XI,NaN,NaN,NaN
2,3869685,1,0,0,France,Half Start,NaN,NaN,NaN
3,3869685,1,0,0,Argentina,Half Start,NaN,NaN,NaN
4,3869685,2,45,0,France,Half Start,NaN,NaN,NaN


Filtro los goles reales de la final.


In [96]:
df_goles_final = df_eventos_final[
    (df_eventos_final["period"] <= 4) &
    (df_eventos_final["type"] == "Shot") &
    (df_eventos_final["shot_outcome"] == "Goal") &
    (df_eventos_final["team"].isin(["Argentina", "France"]))
].copy()

df_goles_final["second"] = df_goles_final["second"].fillna(0)

df_goles_final["minuto_juego"] = (
    df_goles_final["minute"] +
    df_goles_final["second"] / 60
)

df_goles_final = df_goles_final.sort_values(
    ["period", "minute", "second", "index"]
).reset_index(drop=True)

df_goles_final[
    ["team", "player", "minute", "second", "minuto_juego", "shot_type", "shot_outcome"]
]


,team,player,minute,second,minuto_juego,shot_type,shot_outcome
0,Argentina,Lionel Andrés Messi Cuccittini,22,24,22.400000,Penalty,Goal
1,Argentina,Ángel Fabián Di María Hernández,35,22,35.366667,Open Play,Goal
2,France,Kylian Mbappé Lottin,79,24,79.400000,Penalty,Goal
3,France,Kylian Mbappé Lottin,80,59,80.983333,Open Play,Goal
4,Argentina,Lionel Andrés Messi Cuccittini,107,58,107.966667,Open Play,Goal
5,France,Kylian Mbappé Lottin,117,5,117.083333,Penalty,Goal


Creo etiquetas cortas para los goles.


In [97]:
def nombre_corto_jugador(nombre):
    if pd.isna(nombre):
        return ""

    nombre = str(nombre)

    if "Messi" in nombre:
        return "Messi"
    elif "Di María" in nombre or "María" in nombre:
        return "Di María"
    elif "Mbappé" in nombre or "Mbappe" in nombre:
        return "Mbappé"
    else:
        return nombre.split()[-1]

df_goles_final["jugador_corto"] = df_goles_final["player"].apply(nombre_corto_jugador)

df_goles_final["texto_gol"] = (df_goles_final["jugador_corto"] + " · " + df_goles_final["minute"].astype(int).astype(str) + "'")

df_goles_final[["team", "player", "jugador_corto", "texto_gol"]]


,team,player,jugador_corto,texto_gol
0,Argentina,Lionel Andrés Messi Cuccittini,Messi,Messi · 22'
1,Argentina,Ángel Fabián Di María Hernández,Di María,Di María · 35'
2,France,Kylian Mbappé Lottin,Mbappé,Mbappé · 79'
3,France,Kylian Mbappé Lottin,Mbappé,Mbappé · 80'
4,Argentina,Lionel Andrés Messi Cuccittini,Messi,Messi · 107'
5,France,Kylian Mbappé Lottin,Mbappé,Mbappé · 117'


Preparo la base temporal de PE5.


In [98]:
df_timeline_base = df_final[df_final["team"].isin(["Argentina", "France"])].copy()

df_timeline_base["second"] = df_timeline_base["second"].fillna(0)

df_timeline_base["minuto_juego"] = (df_timeline_base["minute"] + df_timeline_base["second"] / 60)

df_timeline_base["incremento_pe5"] = (df_timeline_base["incremento_pe5"].fillna(0))

df_timeline_base = df_timeline_base.sort_values(["team", "period", "minute", "second", "index"]).reset_index(drop=True)

df_timeline_base[["team", "period", "minute", "second", "minuto_juego", "player", "type", "incremento_pe5"]].head()


,team,period,minute,second,minuto_juego,player,type,incremento_pe5
0,Argentina,1,0,16,0.266667,Nahuel Molina Lucero,Pass,0.001137
1,Argentina,1,0,17,0.283333,Rodrigo Javier De Paul,Carry,0.000000
2,Argentina,1,0,31,0.516667,Rodrigo Javier De Paul,Pass,0.001597
3,Argentina,1,0,33,0.550000,Cristian Gabriel Romero,Carry,-0.001597
4,Argentina,1,0,34,0.566667,Cristian Gabriel Romero,Pass,0.001079


Agrupo PE5 por momento de juego.


In [99]:
df_timeline_momento = (df_timeline_base.groupby(["team", "period", "minute", "second", "minuto_juego"],as_index=False).agg(incremento_pe5=("incremento_pe5", "sum")))

df_timeline_momento = df_timeline_momento.sort_values(["team", "period", "minute", "second"]).reset_index(drop=True)

df_timeline_momento["pe5_acumulado"] = (df_timeline_momento.groupby("team")["incremento_pe5"].cumsum())

df_timeline_momento.head()


,team,period,minute,second,minuto_juego,incremento_pe5,pe5_acumulado
0,Argentina,1,0,16,0.266667,0.001137,0.001137
1,Argentina,1,0,17,0.283333,0.000000,0.001137
2,Argentina,1,0,31,0.516667,0.001597,0.002735
3,Argentina,1,0,33,0.550000,-0.001597,0.001137
4,Argentina,1,0,34,0.566667,0.001079,0.002216


Calculo el minuto final del partido.


In [100]:
minuto_final = (
    df_eventos_final[df_eventos_final["period"] <= 4]["minute"] +
    df_eventos_final[df_eventos_final["period"] <= 4]["second"].fillna(0) / 60).max()

minuto_final = float(minuto_final) + 1


Agrego puntos de inicio y cierre temporal.


In [101]:
df_inicio = pd.DataFrame({
    "team": ["Argentina", "France"],
    "period": [1, 1],
    "minute": [0, 0],
    "second": [0, 0],
    "minuto_juego": [0.0, 0.0],
    "incremento_pe5": [0.0, 0.0],
    "pe5_acumulado": [0.0, 0.0]})

df_cierre = (
    df_timeline_momento
    .sort_values(["team", "period", "minute", "second"])
    .groupby("team")
    .tail(1)
    .copy())

df_cierre["period"] = 4
df_cierre["minute"] = int(minuto_final)
df_cierre["second"] = 0
df_cierre["minuto_juego"] = minuto_final
df_cierre["incremento_pe5"] = 0

df_timeline_plot = pd.concat(
    [df_inicio, df_timeline_momento, df_cierre],
    ignore_index=True)

df_timeline_plot = df_timeline_plot.sort_values(
    ["team", "period", "minute", "second", "minuto_juego"]
).reset_index(drop=True)

df_timeline_plot["pe5_acumulado"] = (df_timeline_plot.groupby("team")["pe5_acumulado"].ffill().fillna(0))


In [102]:
df_timeline_plot.groupby("team").agg(
    minuto_inicio=("minuto_juego", "min"),
    minuto_fin=("minuto_juego", "max"),
    puntos=("minuto_juego", "count"),
    pe5_final=("pe5_acumulado", "last"))


,minuto_inicio,minuto_fin,puntos,pe5_final
team,,,,
Argentina,0.0,125.116667,983,1.669991
France,0.0,125.116667,762,1.042347


Ubico los goles sobre la curva PE5.


In [103]:
df_goles_plot_lista = []

for equipo in ["Argentina", "France"]:

    df_linea_equipo = (
        df_timeline_plot[df_timeline_plot["team"] == equipo]
        [["minuto_juego", "pe5_acumulado"]]
        .sort_values("minuto_juego")
        .copy())

    df_goles_equipo = (
        df_goles_final[df_goles_final["team"] == equipo]
        .sort_values("minuto_juego")
        .copy())

    df_goles_equipo = pd.merge_asof(
        df_goles_equipo,
        df_linea_equipo,
        on="minuto_juego",
        direction="backward")

    df_goles_equipo = df_goles_equipo.rename(
        columns={"pe5_acumulado": "pe5_en_gol"})

    df_goles_plot_lista.append(df_goles_equipo)

df_goles_plot = pd.concat(
    df_goles_plot_lista,
    ignore_index=True)

df_goles_plot["pe5_en_gol"] = df_goles_plot["pe5_en_gol"].fillna(0)

df_goles_plot[["team", "player", "texto_gol", "minute", "minuto_juego", "pe5_en_gol", "shot_type"]]


,team,player,texto_gol,minute,minuto_juego,pe5_en_gol,shot_type
0,Argentina,Lionel Andrés Messi Cuccittini,Messi · 22',22,22.400000,0.210335,Penalty
1,Argentina,Ángel Fabián Di María Hernández,Di María · 35',35,35.366667,0.490824,Open Play
2,Argentina,Lionel Andrés Messi Cuccittini,Messi · 107',107,107.966667,1.430449,Open Play
3,France,Kylian Mbappé Lottin,Mbappé · 79',79,79.400000,0.526304,Penalty
4,France,Kylian Mbappé Lottin,Mbappé · 80',80,80.983333,0.614492,Open Play
5,France,Kylian Mbappé Lottin,Mbappé · 117',117,117.083333,0.908193,Penalty


Ajusto posiciones de etiquetas de goles.


In [104]:
df_goles_plot = df_goles_plot.sort_values(
    ["team", "minuto_juego"]
).reset_index(drop=True)

df_goles_plot["n_gol_equipo"] = (
    df_goles_plot
    .groupby("team")
    .cumcount() + 1)

df_goles_plot["ax"] = 0
df_goles_plot["ay"] = -45

df_goles_plot.loc[
    (df_goles_plot["team"] == "Argentina") &
    (df_goles_plot["n_gol_equipo"] == 1),
    ["ax", "ay"]
] = [-45, -45]

df_goles_plot.loc[
    (df_goles_plot["team"] == "Argentina") &
    (df_goles_plot["n_gol_equipo"] == 2),
    ["ax", "ay"]
] = [55, -45]

df_goles_plot.loc[
    (df_goles_plot["team"] == "Argentina") &
    (df_goles_plot["n_gol_equipo"] == 3),
    ["ax", "ay"]
] = [-55, -55]

df_goles_plot.loc[
    (df_goles_plot["team"] == "France") &
    (df_goles_plot["n_gol_equipo"] == 1),
    ["ax", "ay"]
] = [-70, -35]

df_goles_plot.loc[
    (df_goles_plot["team"] == "France") &
    (df_goles_plot["n_gol_equipo"] == 2),
    ["ax", "ay"]
] = [75, -55]

df_goles_plot.loc[
    (df_goles_plot["team"] == "France") &
    (df_goles_plot["n_gol_equipo"] == 3),
    ["ax", "ay"]
] = [-60, -55]

df_goles_plot[["team", "texto_gol", "minuto_juego", "pe5_en_gol", "ax", "ay"]]


,team,texto_gol,minuto_juego,pe5_en_gol,ax,ay
0,Argentina,Messi · 22',22.400000,0.210335,-45,-45
1,Argentina,Di María · 35',35.366667,0.490824,55,-45
2,Argentina,Messi · 107',107.966667,1.430449,-55,-55
3,France,Mbappé · 79',79.400000,0.526304,-70,-35
4,France,Mbappé · 80',80.983333,0.614492,75,-55
5,France,Mbappé · 117',117.083333,0.908193,-60,-55


Grafico PE5 acumulado con goles.


In [110]:
colores_equipo = {
    "Argentina": "#6EC6FF",
    "France": "#EF4135"}

rellenos_equipo = {
    "Argentina": "rgba(110, 198, 255, 0.10)",
    "France": "rgba(239, 65, 53, 0.08)"}

fig = go.Figure()

for equipo in ["Argentina", "France"]:

    df_equipo = (
        df_timeline_plot[df_timeline_plot["team"] == equipo]
        .sort_values("minuto_juego")
        .copy()
    )

    fig.add_trace(
        go.Scatter(
            x=df_equipo["minuto_juego"],
            y=df_equipo["pe5_acumulado"],
            mode="lines",
            name=equipo,
            line=dict(
                color=colores_equipo[equipo],
                width=4,
                shape="hv"
            ),
            connectgaps=True,
            fill="tozeroy",
            fillcolor=rellenos_equipo[equipo],
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Minuto: %{x:.1f}<br>"
                "PE5 acumulado: %{y:.3f}<extra></extra>")))

for equipo in ["Argentina", "France"]:

    df_goles_equipo = (
        df_goles_plot[df_goles_plot["team"] == equipo]
        .sort_values("minuto_juego")
        .copy())

    fig.add_trace(
        go.Scatter(
            x=df_goles_equipo["minuto_juego"],
            y=df_goles_equipo["pe5_en_gol"],
            mode="markers",
            name=f"Goles {equipo}",
            marker=dict(
                size=18,
                color=colores_equipo[equipo],
                line=dict(
                    color="#FFFFFF",
                    width=2
                ),
                symbol="star-diamond"
            ),
            customdata=df_goles_equipo[["player", "texto_gol"]],
            hovertemplate=(
                "<b>Gol</b><br>"
                "Jugador: %{customdata[0]}<br>"
                "Minuto: %{x:.1f}<br>"
                "PE5 acumulado: %{y:.3f}<extra></extra>")))

for _, gol in df_goles_plot.iterrows():

    equipo = gol["team"]

    fig.add_annotation(
        x=gol["minuto_juego"],
        y=gol["pe5_en_gol"],
        text=f"<b>{gol['texto_gol']}</b>",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1.2,
        arrowcolor=colores_equipo[equipo],
        ax=gol["ax"],
        ay=gol["ay"],
        bgcolor="rgba(6, 19, 39, 0.88)",
        bordercolor=colores_equipo[equipo],
        borderwidth=1.2,
        borderpad=5,
        font=dict(
            size=11,
            color="#F8FAFC",
            family="Outfit, Arial, sans-serif"))

fig.update_layout(
    title=dict(
        text="<b>Argentina vs Francia — PE5 acumumulado - Final</b>",
        x=0.5,
        xanchor="center",
        font=dict(
            family="Outfit, Arial, sans-serif",
            size=36,
            color="#F8FAFC"
        )
    ),
    paper_bgcolor="#061327",
    plot_bgcolor="#0B1B33",
    font=dict(
        color="#E5E7EB",
        family="Outfit, Arial, sans-serif"),
    height=720,
    margin=dict(l=85, r=75, t=120, b=165),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(0,0,0,0)",
        font=dict(
            size=13,
            color="#E5E7EB",
            family="Outfit, Arial, sans-serif")),
    hovermode="x unified")

fig.update_xaxes(
    title_text="<b>Minuto de juego</b>",
    range=[0, minuto_final],
    title_standoff=22,
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    ticks="outside",
    tickfont=dict(
        size=12,
        color="#CBD5E1",
        family="Outfit, Arial, sans-serif"),
    title_font=dict(
        size=15,
        color="#F8FAFC",
        family="Outfit, Arial, sans-serif"))

fig.update_yaxes(
    title_text="<b>PE5 acumulado</b>",
    title_standoff=18,
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    tickfont=dict(
        size=12,
        color="#CBD5E1",
        family="Outfit, Arial, sans-serif"),
    title_font=dict(
        size=15,
        color="#F8FAFC",
        family="Outfit, Arial, sans-serif"))

fig.add_annotation(
    x=0.5,
    y=-0.24,
    xref="paper",
    yref="paper",
    text=(
        "Línea escalonada = acumulado de incremento PE5 en pases y conducciones · "
        "Estrellas = goles durante el tiempo de juego"),
    showarrow=False,
    font=dict(
        size=14,
        color="#9FB3C8",
        family="Outfit, Arial, sans-serif"),
    align="center")

fig.show()


Grafico las acciones que más aumentaron PE5.


In [112]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

df_mapa = df_final[
    (df_final["team"].isin(["Argentina", "France"])) &
    (df_final["incremento_pe5"] > 0) &
    (df_final["x"].notna()) &
    (df_final["y"].notna()) &
    (df_final["x_fin"].notna()) &
    (df_final["y_fin"].notna())
].copy()

df_mapa["second"] = df_mapa["second"].fillna(0)
df_mapa["minuto_juego"] = df_mapa["minute"] + df_mapa["second"] / 60

def nombre_corto_jugador(nombre):
    if pd.isna(nombre):
        return ""

    nombre = str(nombre)

    if "Messi" in nombre:
        return "Messi"
    elif "Di María" in nombre or "María" in nombre:
        return "Di María"
    elif "Mbappé" in nombre or "Mbappe" in nombre:
        return "Mbappé"
    elif "Mac Allister" in nombre:
        return "Mac Allister"
    elif "Griezmann" in nombre:
        return "Griezmann"
    else:
        return nombre.split()[-1]

df_mapa["jugador_corto"] = df_mapa["player"].apply(nombre_corto_jugador)

top_n_por_equipo = 6

df_top = (
    df_mapa
    .sort_values(["team", "incremento_pe5"], ascending=[True, False])
    .groupby("team")
    .head(top_n_por_equipo)
    .copy()
)

df_top = df_top.sort_values(
    ["team", "incremento_pe5"],
    ascending=[True, False]
).reset_index(drop=True)

df_top["rank_equipo"] = df_top.groupby("team").cumcount() + 1
df_top["mostrar_etiqueta"] = df_top["rank_equipo"] <= 3

df_top["texto_label"] = (
    df_top["jugador_corto"] +
    " · " +
    df_top["type"] +
    "<br>+" +
    df_top["incremento_pe5"].round(3).astype(str)
)

df_top["x_plot"] = df_top["x"]
df_top["y_plot"] = df_top["y"]
df_top["x_fin_plot"] = df_top["x_fin"]
df_top["y_fin_plot"] = df_top["y_fin"]

mask_fr = df_top["team"] == "France"

df_top.loc[mask_fr, "x_plot"] = 120 - df_top.loc[mask_fr, "x"]
df_top.loc[mask_fr, "x_fin_plot"] = 120 - df_top.loc[mask_fr, "x_fin"]

colores_equipo = {
    "Argentina": "#6EC6FF",
    "France": "#EF4135"
}

fig = go.Figure()

color_fondo = "#061327"
color_cancha = "#0B1B33"
color_lineas = "rgba(255,255,255,0.55)"

shapes_pitch = [
    dict(type="rect", x0=0, y0=0, x1=120, y1=80, line=dict(color=color_lineas, width=2)),
    dict(type="line", x0=60, y0=0, x1=60, y1=80, line=dict(color=color_lineas, width=2)),
    dict(type="circle", x0=50, y0=30, x1=70, y1=50, line=dict(color=color_lineas, width=2)),

    dict(type="rect", x0=0, y0=18, x1=18, y1=62, line=dict(color=color_lineas, width=2)),
    dict(type="rect", x0=0, y0=30, x1=6, y1=50, line=dict(color=color_lineas, width=2)),

    dict(type="rect", x0=102, y0=18, x1=120, y1=62, line=dict(color=color_lineas, width=2)),
    dict(type="rect", x0=114, y0=30, x1=120, y1=50, line=dict(color=color_lineas, width=2)),

    dict(type="circle", x0=11, y0=39, x1=13, y1=41, line=dict(color=color_lineas, width=2), fillcolor=color_lineas),
    dict(type="circle", x0=107, y0=39, x1=109, y1=41, line=dict(color=color_lineas, width=2), fillcolor=color_lineas),
    dict(type="circle", x0=59, y0=39, x1=61, y1=41, line=dict(color=color_lineas, width=2), fillcolor=color_lineas),
]

fig.update_layout(
    paper_bgcolor=color_fondo,
    plot_bgcolor=color_cancha,
    shapes=shapes_pitch
)

for _, fila in df_top.iterrows():

    equipo = fila["team"]
    color = colores_equipo[equipo]

    x0 = fila["x_plot"]
    y0 = fila["y_plot"]
    x1 = fila["x_fin_plot"]
    y1 = fila["y_fin_plot"]

    fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref="x", yref="y", axref="x", ayref="y", showarrow=True, arrowhead=3, 
                       arrowsize=1.2, arrowwidth=2.6, arrowcolor=color, opacity=0.95)

    fig.add_trace(
        go.Scatter(
            x=[x0],
            y=[y0],
            mode="markers",
            marker=dict(
                size=8,
                color=color,
                line=dict(color="#FFFFFF", width=1)
            ),
            customdata=[[
                fila["player"],
                fila["team"],
                fila["type"],
                fila["minute"],
                fila["incremento_pe5"]
            ]],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Equipo: %{customdata[1]}<br>"
                "Acción: %{customdata[2]}<br>"
                "Minuto: %{customdata[3]}<br>"
                "Incremento PE5: %{customdata[4]:.3f}<extra></extra>"
            ),
            showlegend=False))

    fig.add_trace(
        go.Scatter(
            x=[x1],
            y=[y1],
            mode="markers",
            marker=dict(
                size=10,
                color=color,
                symbol="diamond",
                line=dict(color="#FFFFFF", width=1.2)),
            customdata=[[
                fila["player"],
                fila["team"],
                fila["type"],
                fila["minute"],
                fila["incremento_pe5"]]],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Equipo: %{customdata[1]}<br>"
                "Acción: %{customdata[2]}<br>"
                "Minuto: %{customdata[3]}<br>"
                "Incremento PE5: %{customdata[4]:.3f}<extra></extra>"),
            showlegend=False))

for _, fila in df_top[df_top["mostrar_etiqueta"]].iterrows():

    equipo = fila["team"]
    color = colores_equipo[equipo]

    if equipo == "Argentina":
        xshift = 10
        xanchor = "left"
    else:
        xshift = -10
        xanchor = "right"

    fig.add_annotation(
        x=fila["x_plot"],
        y=fila["y_plot"],
        text=f"<b>{fila['texto_label']}</b>",
        showarrow=False,
        xanchor=xanchor,
        yanchor="bottom",
        xshift=xshift,
        yshift=10,
        bgcolor="rgba(6, 19, 39, 0.88)",
        bordercolor=color,
        borderwidth=1.2,
        borderpad=4,
        font=dict(
            size=11,
            color="#F8FAFC",
            family="Outfit, Arial, sans-serif"))

for equipo in ["Argentina", "France"]:
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            marker=dict(size=10, color=colores_equipo[equipo]),
            name=equipo))

fig.add_annotation(
    x=108,
    y=6,
    text="<b>Argentina →</b>",
    showarrow=False,
    font=dict(
        size=14,
        color=colores_equipo["Argentina"],
        family="Outfit, Arial, sans-serif"))

fig.add_annotation(
    x=12,
    y=6,
    text="<b>← France</b>",
    showarrow=False,
    font=dict(
        size=14,
        color=colores_equipo["France"],
        family="Outfit, Arial, sans-serif"))

fig.update_layout(
    title=dict(
        text="<b>Final Argentina vs Francia — acciones que más aumentaron el PE5</b>",
        x=0.5,
        xanchor="center",
        font=dict(
            family="Outfit, Arial, sans-serif",
            size=30,
            color="#F8FAFC")),
    font=dict(
        color="#E5E7EB",
        family="Outfit, Arial, sans-serif"),
    height=720,
    margin=dict(l=40, r=40, t=110, b=120),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(0,0,0,0)",
        font=dict(
            size=13,
            color="#E5E7EB",
            family="Outfit, Arial, sans-serif")))

fig.update_xaxes(
    range=[0, 120],
    showgrid=False,
    zeroline=False,
    showticklabels=False,
    visible=False)

fig.update_yaxes(
    range=[80, 0],
    showgrid=False,
    zeroline=False,
    showticklabels=False,
    visible=False,
    scaleanchor="x",
    scaleratio=1)

fig.add_annotation(
    x=0.5,
    y=-0.08,
    xref="paper",
    yref="paper",
    text=(
        "Flechas = acciones con mayor incremento PE5 en la final · "
        "Punto inicial = origen · Rombo = destino · "
        "Etiquetas visibles = top 3 por equipo"),
    showarrow=False,
    font=dict(
        size=12,
        color="#9FB3C8",
        family="Outfit, Arial, sans-serif"),
    align="center")

fig.show()


Calculo momentum PE5 por tramos.


In [113]:
ventana_minutos = 5

df_momentum = df_final[
    df_final["team"].isin(["Argentina", "France"])
].copy()

df_momentum["second"] = df_momentum["second"].fillna(0)

df_momentum["minuto_juego"] = (
    df_momentum["minute"] +
    df_momentum["second"] / 60
)

df_momentum["incremento_pe5"] = df_momentum["incremento_pe5"].fillna(0)

df_momentum["tramo_inicio"] = (
    (df_momentum["minuto_juego"] // ventana_minutos) * ventana_minutos
).astype(int)

df_momentum["tramo_fin"] = df_momentum["tramo_inicio"] + ventana_minutos

df_momentum["tramo"] = (
    df_momentum["tramo_inicio"].astype(str) +
    "-" +
    df_momentum["tramo_fin"].astype(str)
)

df_momentum_tramo = (
    df_momentum
    .groupby(["team", "tramo_inicio", "tramo_fin", "tramo"], as_index=False)
    .agg(
        pe5_neto=("incremento_pe5", "sum"),
        pe5_positivo=("pe5_positivo", "sum"),
        pe5_negativo=("pe5_negativo", "sum"),
        acciones=("incremento_pe5", "count")
    )
)

df_momentum_wide = (
    df_momentum_tramo
    .pivot_table(
        index=["tramo_inicio", "tramo_fin", "tramo"],
        columns="team",
        values="pe5_neto",
        fill_value=0
    )
    .reset_index()
)

if "Argentina" not in df_momentum_wide.columns:
    df_momentum_wide["Argentina"] = 0

if "France" not in df_momentum_wide.columns:
    df_momentum_wide["France"] = 0

df_momentum_wide["momentum_pe5"] = (
    df_momentum_wide["Argentina"] -
    df_momentum_wide["France"]
)

df_momentum_wide["equipo_dominante"] = np.where(
    df_momentum_wide["momentum_pe5"] >= 0,
    "Argentina",
    "France"
)

df_momentum_wide["color"] = np.where(
    df_momentum_wide["momentum_pe5"] >= 0,
    "#6EC6FF",
    "#EF4135"
)

df_momentum_wide = df_momentum_wide.sort_values("tramo_inicio").reset_index(drop=True)

df_momentum_wide.head()


team,tramo_inicio,tramo_fin,tramo,Argentina,France,momentum_pe5,equipo_dominante,color
0,0,5,0-5,0.034045,0.000915,0.033130,Argentina,#6EC6FF
1,5,10,5-10,0.017606,0.000600,0.017006,Argentina,#6EC6FF
2,10,15,10-15,0.058278,0.025112,0.033166,Argentina,#6EC6FF
3,15,20,15-20,0.088817,0.023671,0.065146,Argentina,#6EC6FF
4,20,25,20-25,0.011589,0.001027,0.010562,Argentina,#6EC6FF


Grafico momentum PE5 por tramo.


In [127]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=df_momentum_wide["tramo"],
        y=df_momentum_wide["momentum_pe5"],
        marker=dict(
            color=df_momentum_wide["color"],
            line=dict(
                color="rgba(255,255,255,0.45)",
                width=1.2)),
        text=df_momentum_wide["momentum_pe5"],
        texttemplate="<b>%{text:.3f}</b>",
        textposition="outside",
        textfont=dict(
            size=12,
            color="#F8FAFC",
            family="Outfit, Arial, sans-serif"),
        hovertemplate=(
            "<b>Tramo %{x}</b><br>"
            "Momentum PE5: %{y:.3f}<br>"
            "<extra></extra>")))

fig.add_hline(
    y=0,
    line_width=1.5,
    line_color="rgba(255,255,255,0.55)")

fig.update_layout(
    title=dict(
        text="<b>Argentina vs Francia — momentum PE5 por tramos (5 min)</b>",
        x=0.5,
        xanchor="center",
        font=dict(
            family="Outfit, Arial, sans-serif",
            size=36,
            color="#F8FAFC")),
    paper_bgcolor="#061327",
    plot_bgcolor="#0B1B33",
    font=dict(
        color="#E5E7EB",
        family="Outfit, Arial, sans-serif"),
    height=650,
    margin=dict(l=80, r=70, t=120, b=140),
    showlegend=False)

fig.update_xaxes(
    title_text="<b>Tramo del partido</b>",
    title_standoff=18,
    showgrid=False,
    zeroline=False,
    ticks="outside",
    tickangle=-45,
    tickfont=dict(
        size=11,
        color="#CBD5E1",
        family="Outfit, Arial, sans-serif"),
    title_font=dict(
        size=15,
        color="#F8FAFC",
        family="Outfit, Arial, sans-serif"))

fig.update_yaxes(
    title_text="<b>Momentum PE5<br>Argentina − Francia</b>",
    title_standoff=16,
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    tickfont=dict(
        size=12,
        color="#CBD5E1",
        family="Outfit, Arial, sans-serif"),
    title_font=dict(
        size=15,
        color="#F8FAFC",
        family="Outfit, Arial, sans-serif"))

fig.add_annotation(
    x=0.02,
    y=1.08,
    xref="paper",
    yref="paper",
    text="<b style='color:#6EC6FF'>Barras positivas: dominio territorial PE5 de Argentina</b>",
    showarrow=False,
    align="left",
    font=dict(
        size=15,
        color="#6EC6FF",
        family="Outfit, Arial, sans-serif"))

fig.add_annotation(
    x=0.98,
    y=1.08,
    xref="paper",
    yref="paper",
    text="<b style='color:#EF4135'>Barras negativas: dominio territorial PE5 de Francia</b>",
    showarrow=False,
    align="right",
    font=dict(
        size=15,
        color="#EF4135",
        family="Outfit, Arial, sans-serif"))

fig.add_annotation(
    x=0.5,
    y=-0.27,
    xref="paper",
    yref="paper",
    text=(
        "Momentum PE5 = PE5 neto de Argentina menos PE5 neto de Francia en cada tramo. "
        "El PE5 surge de acciones valoradas por cambio de peligro zonal."),
    showarrow=False,
    font=dict(
        size=14,
        color="#9FB3C8",
        family="Outfit, Arial, sans-serif"),
    align="center")

fig.show()


In [138]:
ventana_minutos = 5

df_heat = df_final[df_final["team"].isin(["Argentina", "France"])].copy()

df_heat["second"] = df_heat["second"].fillna(0)
df_heat["minuto_juego"] = df_heat["minute"] + df_heat["second"] / 60
df_heat["tramo"] = ((df_heat["minuto_juego"] // ventana_minutos) * ventana_minutos).astype(int)
df_heat["tramo_label"] = df_heat["tramo"].astype(str) + "-" + (df_heat["tramo"] + ventana_minutos).astype(str)
df_heat["incremento_pe5"] = df_heat["incremento_pe5"].fillna(0)
df_heat["jugador_corto"] = df_heat["player"].apply(nombre_corto_jugador)

df_heat_player = (
    df_heat
    .groupby(["team", "player", "jugador_corto", "tramo", "tramo_label"], as_index=False)
    .agg(pe5=("incremento_pe5", "sum")))

top_jugadores = (
    df_heat_player
    .groupby(["player", "jugador_corto"], as_index=False)
    .agg(pe5_total=("pe5", "sum"))
    .sort_values("pe5_total", ascending=False).head(12))

df_heat_plot = df_heat_player[df_heat_player["player"].isin(top_jugadores["player"])].copy()

fig = px.density_heatmap(
    df_heat_plot,
    x="tramo_label",
    y="jugador_corto",
    z="pe5",
    histfunc="sum",
    color_continuous_scale="magma",
    title="Final Argentina vs Francia — calor PE5 por jugador y tramo")

fig.update_layout(
    title=dict(
        text="<b>Final Argentina vs Francia — PE5 por jugador y tramo</b><br>"
             "<span style='font-size:15px;color:#9FB3C8'>Suma de incremento PE5 en ventanas de 5 minutos</span>",
        x=0.5,
        xanchor="center",
        font=dict(size=40, color="#F8FAFC", family="Outfit, Arial, sans-serif")),
    paper_bgcolor="#061327",
    plot_bgcolor="#0B1B33",
    font=dict(color="#E5E7EB", family="Outfit, Arial, sans-serif"),
    height=680,
    margin=dict(l=120, r=80, t=120, b=110))

fig.update_xaxes(title_text="<b>Tramo del partido</b>", tickangle=-45)
fig.update_yaxes(title_text="")
fig.update_coloraxes(colorbar_title="PE5")

fig.show()

In [133]:
df_player_line = df_final[
    df_final["team"].isin(["Argentina", "France"])
].copy()

df_player_line["second"] = df_player_line["second"].fillna(0)
df_player_line["minuto_juego"] = df_player_line["minute"] + df_player_line["second"] / 60
df_player_line["incremento_pe5"] = df_player_line["incremento_pe5"].fillna(0)
df_player_line["jugador_corto"] = df_player_line["player"].apply(nombre_corto_jugador)

top_players = (
    df_player_line
    .groupby(["player", "jugador_corto", "team"], as_index=False)
    .agg(pe5_total=("incremento_pe5", "sum"))
    .sort_values("pe5_total", ascending=False)
    .head(8))

df_player_line = df_player_line[
    df_player_line["player"].isin(top_players["player"])
].copy()

df_player_line = df_player_line.sort_values(["player", "minuto_juego"])

df_player_line["pe5_acumulado"] = (
    df_player_line
    .groupby("player")["incremento_pe5"]
    .cumsum())

colores_equipo = {
    "Argentina": "#6EC6FF",
    "France": "#EF4135"}

fig = px.line(
    df_player_line,
    x="minuto_juego",
    y="pe5_acumulado",
    facet_col="jugador_corto",
    facet_col_wrap=4,
    color="team",
    color_discrete_map=colores_equipo,
    title="Final Argentina vs Francia — PE5 acumulado individual")

fig.update_traces(line=dict(width=3))

fig.update_layout(
    title=dict(
        text="<b>Final Argentina vs Francia — PE5 acumulado individual</b><br>"
             "<span style='font-size:15px;color:#9FB3C8'>Top 8 jugadores por PE5 total en pases y conducciones</span>",
        x=0.5,
        xanchor="center",
        font=dict(size=28, color="#F8FAFC", family="Outfit, Arial, sans-serif")),
    paper_bgcolor="#061327",
    plot_bgcolor="#0B1B33",
    font=dict(color="#E5E7EB", family="Outfit, Arial, sans-serif"),
    height=720,
    margin=dict(l=70, r=70, t=130, b=80),
    legend=dict(
        orientation="h",
        y=1.03,
        x=0.5,
        xanchor="center"))

fig.update_xaxes(title_text="Minuto", gridcolor="rgba(255,255,255,0.08)")
fig.update_yaxes(title_text="PE5 acumulado", gridcolor="rgba(255,255,255,0.08)")

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [156]:
df_player_line = df_final[df_final["team"].isin(["Argentina", "France"])].copy()

df_player_line["second"] = df_player_line["second"].fillna(0)
df_player_line["minuto_juego"] = (
    df_player_line["minute"] +
    df_player_line["second"] / 60)
df_player_line["incremento_pe5"] = df_player_line["incremento_pe5"].fillna(0)
df_player_line["jugador_corto"] = df_player_line["player"].apply(nombre_corto_jugador)

top_players = (
    df_player_line
    .groupby(["player", "jugador_corto", "team"], as_index=False)
    .agg(pe5_total=("incremento_pe5", "sum"))
    .sort_values("pe5_total", ascending=False)
    .head(12))

df_player_line = df_player_line[
    df_player_line["player"].isin(top_players["player"])].copy()

df_player_line = df_player_line.sort_values(["player", "minuto_juego"])

df_player_line["pe5_acumulado"] = (
    df_player_line
    .groupby("player")["incremento_pe5"]
    .cumsum())

orden_jugadores = top_players["jugador_corto"].tolist()

df_player_line["jugador_corto"] = pd.Categorical(
    df_player_line["jugador_corto"],
    categories=orden_jugadores,
    ordered=True)

colores_equipo = {
    "Argentina": "#6EC6FF",
    "France": "#EF4135"}

y_min = min(0, df_player_line["pe5_acumulado"].min())
y_max = df_player_line["pe5_acumulado"].max() * 1.08

fig = px.line(
    df_player_line,
    x="minuto_juego",
    y="pe5_acumulado",
    facet_col="jugador_corto",
    facet_col_wrap=4,
    color="team",
    color_discrete_map=colores_equipo,
    category_orders={"jugador_corto": orden_jugadores},
    line_shape="hv",
    facet_row_spacing=0.09,
    facet_col_spacing=0.04)

fig.update_traces(
    line=dict(width=3.9),
    opacity=0.96,
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Minuto: %{x:.1f}<br>"
        "PE5 acumulado: %{y:.3f}<extra></extra>"))

fig.update_layout(
    title=dict(
        text="<b>Final Argentina vs Francia — PE5 acumulado individual</b><br>"
             "<span style='font-size:20px;color:#9FB3C8'>Top 12 jugadores por PE5 total en pases y conducciones</span>",
        x=0.5,
        xanchor="center",
        y=0.95,
        font=dict(
            size=40,
            color="#F8FAFC",
            family="Outfit, Arial, sans-serif")),
    paper_bgcolor="#061327",
    plot_bgcolor="#0B1B33",
    font=dict(
        color="#E5E7EB",
        family="Outfit, Arial, sans-serif"),
    height=980,
    margin=dict(l=80, r=70, t=170, b=90),
    legend=dict(
        title_text="",
        orientation="h",
        yanchor="top",
        y=1.06,
        x=0.5,
        xanchor="center",
        bgcolor="rgba(0,0,0,0)",
        font=dict(
            size=13,
            color="#E5E7EB",
            family="Outfit, Arial, sans-serif")))

fig.update_xaxes(
    title_text="",
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=False,
    ticks="outside",
    tickfont=dict(
        size=10,
        color="#CBD5E1",
        family="Outfit, Arial, sans-serif"),
    showline=True,
    linecolor="rgba(255,255,255,0.18)",
    linewidth=1)

fig.update_yaxes(
    title_text="",
    range=[y_min, y_max],
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zeroline=True,
    zerolinecolor="rgba(255,255,255,0.25)",
    zerolinewidth=1.1,
    tickfont=dict(
        size=10,
        color="#CBD5E1",
        family="Outfit, Arial, sans-serif"),
    showline=True,
    linecolor="rgba(255,255,255,0.18)",
    linewidth=1)

fig.for_each_annotation(
    lambda a: a.update(
        text=f"<b>{a.text.split('=')[-1]}</b>",
        font=dict(
            size=16,
            color="#F8FAFC",
            family="Outfit, Arial, sans-serif"),
        yshift=6))

fig.add_annotation(
    x=0.5,
    y=-0.055,
    xref="paper",
    yref="paper",
    text="<b>Minuto del partido</b>",
    showarrow=False,
    font=dict(
        size=14,
        color="#F8FAFC",
        family="Outfit, Arial, sans-serif"))

fig.add_annotation(
    x=-0.045,
    y=0.5,
    xref="paper",
    yref="paper",
    text="<b>PE5 acumulado</b>",
    textangle=-90,
    showarrow=False,
    font=dict(
        size=14,
        color="#F8FAFC",
        family="Outfit, Arial, sans-serif"))

fig.show()